# Noticias con palabras COVID en `tweets_medios_arg_covid`

**Objetivo:** Cuantificar cuántas noticias (artículos) incluyeron palabras relacionadas con COVID-19, tanto en el título/resumen de la noticia como en los comentarios de los usuarios.

**Dataset:** `data/tweets_medios_arg_covid.csv`  
**Columnas clave:**
- `title` / `resumen` — texto de la noticia publicada por el medio
- `text` — comentario del usuario (respuesta al tweet de la noticia)
- `covid` — 1 si el **comentario** contiene palabras COVID (generado por `add_covid_label.py`)
- `tweet_id_noticia` — ID único de cada noticia

## 1. Setup y palabras clave

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Mismas palabras clave usadas en add_covid_label.py
COVID_KEYWORDS = [
    "coronavirus", "covid", "Wuhan", "cuarentena", "normalidad",
    "aislamiento", "encierro", "fase", "infectados", "distanciamiento",
    "fiebre", "síntomas"
]

print("Palabras clave COVID utilizadas:")
for kw in COVID_KEYWORDS:
    print(f"  · {kw}")

## 2. Carga del dataset

In [ ]:
df = pd.read_csv('data/tweets_medios_arg_covid.csv', parse_dates=['date_tweet'])

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Rango temporal:  {df['date_tweet'].min().date()} → {df['date_tweet'].max().date()}")
print(f"Noticias únicas: {df['tweet_id_noticia'].nunique():,}")
df.head(3)

## 3. Noticias con palabras COVID en el título / resumen

Cada fila del dataset es un **comentario**. Para analizar a nivel de **noticia** se trabaja
sobre el subconjunto de filas únicas por `tweet_id_noticia`, usando los campos `title` y `resumen`.

In [ ]:
# Una fila por noticia (el título y resumen se repiten por cada comentario)
df_noticias = df.drop_duplicates(subset='tweet_id_noticia')[[
    'tweet_id_noticia', 'title', 'resumen', 'medio', 'date_tweet'
]].copy()

print(f"Total noticias únicas: {len(df_noticias):,}")

In [ ]:
pattern = "|".join(COVID_KEYWORDS)

# Buscar en title Y en resumen (case-insensitive)
covid_en_title   = df_noticias['title'].str.contains(pattern, case=False, na=False)
covid_en_resumen = df_noticias['resumen'].str.contains(pattern, case=False, na=False)

df_noticias['covid_noticia'] = (covid_en_title | covid_en_resumen).astype(int)

total_noticias      = len(df_noticias)
noticias_covid      = df_noticias['covid_noticia'].sum()
noticias_no_covid   = total_noticias - noticias_covid
pct_covid           = noticias_covid / total_noticias * 100

print("=" * 55)
print("NOTICIAS CON PALABRAS COVID (en título o resumen)")
print("=" * 55)
print(f"  Total noticias:          {total_noticias:>10,}")
print(f"  Con palabras COVID:      {noticias_covid:>10,}  ({pct_covid:.1f}%)")
print(f"  Sin palabras COVID:      {noticias_no_covid:>10,}  ({100 - pct_covid:.1f}%)")

## 4. Comentarios con palabras COVID

La columna `covid` del dataset indica si el **comentario** (no la noticia) contiene palabras COVID.

In [ ]:
total_comentarios   = len(df)
comentarios_covid   = df['covid'].sum()
pct_com_covid       = comentarios_covid / total_comentarios * 100

# Noticias que recibieron al menos un comentario con palabras COVID
noticias_con_com_covid = df[df['covid'] == 1]['tweet_id_noticia'].nunique()
pct_not_com_covid = noticias_con_com_covid / total_noticias * 100

print("=" * 55)
print("COMENTARIOS CON PALABRAS COVID (columna 'covid')")
print("=" * 55)
print(f"  Total comentarios:        {total_comentarios:>10,}")
print(f"  Comentarios con COVID:    {comentarios_covid:>10,}  ({pct_com_covid:.1f}%)")
print()
print("  Noticias con ≥1 comentario COVID:")
print(f"    {noticias_con_com_covid:>10,}  ({pct_not_com_covid:.1f}% de las noticias)")

## 5. Resumen comparativo

In [ ]:
resumen = pd.DataFrame({
    'Dimensión': [
        'Noticias únicas totales',
        'Noticias con COVID en título/resumen',
        'Noticias con ≥1 comentario COVID',
        'Comentarios totales',
        'Comentarios con palabras COVID',
    ],
    'N': [
        total_noticias,
        noticias_covid,
        noticias_con_com_covid,
        total_comentarios,
        comentarios_covid,
    ],
    '%': [
        100.0,
        round(pct_covid, 1),
        round(pct_not_com_covid, 1),
        100.0,
        round(pct_com_covid, 1),
    ]
})

resumen['N'] = resumen['N'].apply(lambda x: f"{x:,}")
resumen['%'] = resumen['%'].apply(lambda x: f"{x:.1f}%")
display(resumen)

## 6. Noticias COVID por medio

In [ ]:
NOMBRES_MEDIOS = {
    'clarincom': 'Clarín', 'LANACION': 'La Nación', 'infobae': 'Infobae',
    'pagina12': 'Página 12', 'cronica': 'Crónica', 'perfilcom': 'Perfil',
    'laderechamedios': 'La Derecha Medios',
    'laderechadiario': 'La Derecha Diario',
    'izquierdadiario': 'Izquierda Diario',
}

por_medio = (
    df_noticias
    .groupby('medio')
    .agg(
        total_noticias=('tweet_id_noticia', 'count'),
        noticias_covid=('covid_noticia', 'sum')
    )
    .assign(pct_covid=lambda x: x['noticias_covid'] / x['total_noticias'] * 100)
    .sort_values('noticias_covid', ascending=False)
    .reset_index()
)

por_medio['nombre'] = por_medio['medio'].map(NOMBRES_MEDIOS).fillna(por_medio['medio'])

display(
    por_medio[['nombre', 'total_noticias', 'noticias_covid', 'pct_covid']]
    .rename(columns={
        'nombre': 'Medio',
        'total_noticias': 'Total noticias',
        'noticias_covid': 'Noticias COVID',
        'pct_covid': '% COVID'
    })
    .style.format({'Total noticias': '{:,}', 'Noticias COVID': '{:,}', '% COVID': '{:.1f}%'})
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: N absoluto
bars1 = axes[0].barh(
    por_medio['nombre'],
    por_medio['noticias_covid'],
    color=sns.color_palette('Blues_d', len(por_medio))
)
axes[0].set_xlabel('Noticias con palabras COVID')
axes[0].set_title('Noticias COVID por medio (N absoluto)')
for bar, val in zip(bars1, por_medio['noticias_covid']):
    axes[0].text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)

# Gráfico 2: Porcentaje
bars2 = axes[1].barh(
    por_medio['nombre'],
    por_medio['pct_covid'],
    color=sns.color_palette('Oranges_d', len(por_medio))
)
axes[1].set_xlabel('% noticias con palabras COVID')
axes[1].set_title('Noticias COVID por medio (% sobre total del medio)')
for bar, val in zip(bars2, por_medio['pct_covid']):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/noticias_covid_por_medio.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evolución temporal de noticias COVID

In [ ]:
# Agrupar por semana
noticias_semana = (
    df_noticias
    .groupby(pd.Grouper(key='date_tweet', freq='W'))
    .agg(
        total=('tweet_id_noticia', 'count'),
        covid=('covid_noticia', 'sum')
    )
    .assign(no_covid=lambda x: x['total'] - x['covid'])
)

fig, ax = plt.subplots(figsize=(14, 5))

# Franjas pandemia
periodos = [
    (pd.Timestamp('2020-03-20'), pd.Timestamp('2020-04-30'), '#e74c3c', 'Máximo — lockdown total'),
    (pd.Timestamp('2020-05-01'), pd.Timestamp('2020-10-31'), '#e67e22', 'Alto — especialmente AMBA'),
    (pd.Timestamp('2021-04-01'), pd.Timestamp('2021-06-30'), '#d4ac0d', 'Medio-alto — 2ª ola'),
]
franja_handles = []
for inicio, fin, color, label in periodos:
    patch = ax.axvspan(inicio, fin, alpha=0.12, color=color, zorder=0, label=label)
    franja_handles.append(patch)

l1, = ax.plot(noticias_semana.index, noticias_semana['covid'],
              color='#e74c3c', linewidth=1.8, label='Noticias COVID')
l2, = ax.plot(noticias_semana.index, noticias_semana['no_covid'],
              color='#2980b9', linewidth=1.4, alpha=0.6, label='Noticias sin COVID')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.set_ylabel('Noticias por semana')
ax.set_title('Evolución temporal: noticias con y sin palabras COVID (semanal)')

ley_lineas = ax.legend(handles=[l1, l2], loc='upper right', fontsize=9)
ax.add_artist(ley_lineas)
ax.legend(handles=franja_handles, loc='upper left', fontsize=8, title='Restricciones COVID-19')

plt.tight_layout()
plt.savefig('outputs/evolucion_temporal_noticias_covid.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Frecuencia de cada palabra clave COVID en títulos de noticias

In [ ]:
kw_counts = {}
for kw in COVID_KEYWORDS:
    n = df_noticias['title'].str.contains(kw, case=False, na=False).sum()
    kw_counts[kw] = n

kw_df = pd.DataFrame.from_dict(kw_counts, orient='index', columns=['noticias'])
kw_df = kw_df.sort_values('noticias', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(kw_df.index, kw_df['noticias'],
               color=sns.color_palette('viridis', len(kw_df)))
for bar, val in zip(bars, kw_df['noticias']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_xlabel('Cantidad de noticias que incluyen la palabra (en título)')
ax.set_title('Frecuencia de cada palabra clave COVID en títulos de noticias')
plt.tight_layout()
plt.savefig('outputs/frecuencia_keywords_covid.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTabla de frecuencias:")
display(kw_df.sort_values('noticias', ascending=False)
           .rename(columns={'noticias': 'N noticias (en título)'}))